In [7]:
import os
import sys
import numpy as np

# Point to the src folder so Jupyter can find our custom libraries
sys.path.append("..")
from src.data_utils import prepare_data
from src.data_augment import augment_training_data
%load_ext autoreload
%autoreload 2

# Load the serialized data from the directory where they were saved
DATA_DIR = os.path.join("..", "data", "processed")
X = np.load(os.path.join(DATA_DIR, "X_features.npy"))
y = np.load(os.path.join(DATA_DIR, "y_labels.npy"))

print(f"Loaded X shape: {X.shape}")
print(f"Loaded y shape: {y.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded X shape: (1440, 20, 125)
Loaded y shape: (1440,)


In [8]:

X_train_final, X_test_final, y_train, y_test, num_classes = prepare_data(X, y)

print("\n--- DEEP LEARNING TENSORS READY ---")
print(f"Final X_train shape for CNN: {X_train_final.shape}")
print(f"Final X_test shape for CNN: {X_test_final.shape}")
print(f"Final y_train shape for CNN: {y_train.shape}")
print(f"Final y_test shape for CNN: {y_test.shape}")
print(f"Total number of target classes: {num_classes}")


--- DEEP LEARNING TENSORS READY ---
Final X_train shape for CNN: (1152, 20, 125, 1)
Final X_test shape for CNN: (288, 20, 125, 1)
Final y_train shape for CNN: (1152, 8)
Final y_test shape for CNN: (288, 8)
Total number of target classes: 8


In [9]:
# Double the training data using SpecAugment
X_train_final, y_train = augment_training_data(X_train_final, y_train)

print(f"Augmented X_train shape for CNN: {X_train_final.shape}")
print(f"Augmented y_train shape for CNN: {y_train.shape}")

Original training size: 1152
New augmented training size: 2304
Augmented X_train shape for CNN: (2304, 20, 125, 1)
Augmented y_train shape for CNN: (2304, 8)


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, BatchNormalization, GlobalAveragePooling2D, Dense

# Define the input shape based on our final tensors
input_shape = (20, 125, 1)

# Initialize the Sequential model
model = Sequential()

# Block 1: Feature Extraction (Detecting lines, edges, and texture in the MFCC)
model.add(Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu', input_shape=input_shape))
model.add(BatchNormalization()) # Adds zero latency to TinyML, stabilizes training
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2)) # Prevents the network from relying too much on any single feature

# Block 2: Deep Feature Extraction (Detecting complex patterns like vocal formants)
model.add(Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(BatchNormalization()) # Adds zero latency to TinyML, stabilizes training
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))

# Block 3: Deepest Feature Extraction
model.add(Conv2D(128, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(BatchNormalization()) # Adds zero latency to TinyML, stabilizes training
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))

# TinyML Hero: Replaces Flatten and the massive Dense(128) layer
model.add(GlobalAveragePooling2D()) 

# Output Layer: 8 neurons for 8 emotions, using Softmax to output percentages
model.add(Dense(8, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Print the architecture
model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 20, 125, 32)       320       
                                                                 
 batch_normalization_3 (Bat  (None, 20, 125, 32)       128       
 chNormalization)                                                
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 10, 62, 32)        0         
 g2D)                                                            
                                                                 
 dropout_3 (Dropout)         (None, 10, 62, 32)        0         
                                                                 
 conv2d_4 (Conv2D)           (None, 10, 62, 64)        18496     
                                                                 
 batch_normalization_4 (Bat  (None, 10, 62, 64)      

In [12]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Define the callbacks
early_stopping = EarlyStopping(
    monitor='val_accuracy', 
    patience=15, 
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    filepath='best_cnn_model_v4.keras', 
    monitor='val_accuracy', 
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', 
    factor=0.5, 
    patience=4, 
    min_lr=0.00001, 
    verbose=1
)

# Start training
print("--- STARTING MODEL TRAINING ---")
history = model.fit(
    X_train_final, y_train,
    validation_data=(X_test_final, y_test),
    epochs=100,
    batch_size=16, # Reduced batch size for better generalization
    callbacks=[early_stopping, model_checkpoint, reduce_lr]
)
print("--- TRAINING COMPLETE ---")

--- STARTING MODEL TRAINING ---
Epoch 1/100


144/144 [==============================] - ETA: 0s - loss: 1.7381 - accuracy: 0.3472
Epoch 1: val_accuracy improved from -inf to 0.22917, saving model to best_cnn_model_v4.keras
144/144 [==============================] - 31s 98ms/step - loss: 1.7381 - accuracy: 0.3472 - val_loss: 1.9497 - val_accuracy: 0.2292 - lr: 0.0010
Epoch 2/100
144/144 [==============================] - ETA: 0s - loss: 1.4228 - accuracy: 0.4800
Epoch 2: val_accuracy improved from 0.22917 to 0.37153, saving model to best_cnn_model_v4.keras
144/144 [==============================] - 10s 69ms/step - loss: 1.4228 - accuracy: 0.4800 - val_loss: 1.7035 - val_accuracy: 0.3715 - lr: 0.0010
Epoch 3/100
144/144 [==============================] - ETA: 0s - loss: 1.2391 - accuracy: 0.5443
Epoch 3: val_accuracy improved from 0.37153 to 0.46875, saving model to best_cnn_model_v4.keras
144/144 [==============================] - 12s 86ms/step - loss: 1.2391 - accuracy: 0.5443 - val_l

In [26]:
from tensorflow.keras.models import load_model

# Load your best original model
best_model = load_model('best_cnn_model.keras')

# Verify it loaded correctly
best_model.summary()

# Test it against your validation data to confirm the 70% accuracy
loss, accuracy = best_model.evaluate(X_test_final, y_test)
print(f"Restored model accuracy: {accuracy * 100:.2f}%")

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_10 (Conv2D)          (None, 20, 125, 32)       320       
                                                                 
 max_pooling2d_9 (MaxPoolin  (None, 10, 62, 32)        0         
 g2D)                                                            
                                                                 
 dropout_13 (Dropout)        (None, 10, 62, 32)        0         
                                                                 
 conv2d_11 (Conv2D)          (None, 10, 62, 64)        18496     
                                                                 
 max_pooling2d_10 (MaxPooli  (None, 5, 31, 64)         0         
 ng2D)                                                           
                                                                 
 dropout_14 (Dropout)        (None, 5, 31, 64)        

In [27]:
# Load your best original model
best_model_v2 = load_model('best_cnn_model_v2.keras')

# Verify it loaded correctly
best_model_v2.summary()

# Test it against your validation data to confirm the 70% accuracy
loss, accuracy = best_model_v2.evaluate(X_test_final, y_test)
print(f"Restored model accuracy: {accuracy * 100:.2f}%")

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_19 (Conv2D)          (None, 20, 125, 32)       320       
                                                                 
 batch_normalization (Batch  (None, 20, 125, 32)       128       
 Normalization)                                                  
                                                                 
 max_pooling2d_18 (MaxPooli  (None, 10, 62, 32)        0         
 ng2D)                                                           
                                                                 
 dropout_25 (Dropout)        (None, 10, 62, 32)        0         
                                                                 
 conv2d_20 (Conv2D)          (None, 10, 62, 64)        18496     
                                                                 
 batch_normalization_1 (Bat  (None, 10, 62, 64)       

In [29]:
# Load your best original model
best_model_v3 = load_model('best_cnn_model_v3.keras')

# Verify it loaded correctly
best_model_v3.summary()

# Test it against your validation data to confirm the 70% accuracy
loss, accuracy = best_model_v3.evaluate(X_test_final, y_test)
print(f"Restored model accuracy: {accuracy * 100:.2f}%")

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_22 (Conv2D)          (None, 20, 125, 32)       320       
                                                                 
 batch_normalization_3 (Bat  (None, 20, 125, 32)       128       
 chNormalization)                                                
                                                                 
 max_pooling2d_21 (MaxPooli  (None, 10, 62, 32)        0         
 ng2D)                                                           
                                                                 
 dropout_28 (Dropout)        (None, 10, 62, 32)        0         
                                                                 
 conv2d_23 (Conv2D)          (None, 10, 62, 64)        18496     
                                                                 
 batch_normalization_4 (Bat  (None, 10, 62, 64)       